# GEMAMX GROMACS - Libreta definitiva por par en Colab

Esta libreta consolida el flujo real que terminamos usando para correr un par proteina-ligando en Colab con GROMACS.

Defaults listos para ejecutar:
- par: `PIK3CA_H1047R__Alpelisib`
- receptor: `PIK3CA_H1047R_clean.pdb`
- ligando: `ligand_input.mol2`
- basename del ligando: `alpelisib`

Si reutilizas esta libreta para otro par, solo cambia la celda de configuracion.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/GEMAMX


In [ ]:
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/GEMAMX')

DEFAULT_RUN_TAG = 'PIK3CA_H1047R__Alpelisib'
DEFAULT_RECEPTOR_PDB = 'PIK3CA_H1047R_clean.pdb'
DEFAULT_LIGAND_INPUT = 'ligand_input.mol2'
DEFAULT_LIGAND_BASENAME = 'alpelisib'
DEFAULT_FORCEFIELD = 'charmm27'
DEFAULT_WATER_MODEL = 'tip3p'

RUN_TAG = DEFAULT_RUN_TAG
RECEPTOR_FILENAME = DEFAULT_RECEPTOR_PDB
LIGAND_INPUT_FILENAME = DEFAULT_LIGAND_INPUT
LIGAND_BASENAME = DEFAULT_LIGAND_BASENAME
FORCEFIELD = DEFAULT_FORCEFIELD
WATER_MODEL = DEFAULT_WATER_MODEL

RUN_DIR = ROOT / 'gromacs' / 'runs' / RUN_TAG
RECEPTOR_PDB = RUN_DIR / RECEPTOR_FILENAME
PROTEIN_ONLY_PDB = RUN_DIR / RECEPTOR_FILENAME.replace('_clean.pdb', '_protein_only.pdb')
FIXED_PDB = RUN_DIR / RECEPTOR_FILENAME.replace('_clean.pdb', '_fixed.pdb')
LIGAND_INPUT = RUN_DIR / LIGAND_INPUT_FILENAME
LIGAND_ACPYPE_DIR = RUN_DIR / f'{LIGAND_BASENAME}.acpype'
LIGAND_GRO = LIGAND_ACPYPE_DIR / f'{LIGAND_BASENAME}_GMX.gro'
LIGAND_ITP_NAME = f'{LIGAND_BASENAME}_GMX.itp'
LIGAND_POSRE_NAME = f'posre_{LIGAND_BASENAME}.itp'

RUN_DIR.mkdir(parents=True, exist_ok=True)
print('RUN_DIR =', RUN_DIR)
print('RECEPTOR_PDB =', RECEPTOR_PDB)
print('LIGAND_INPUT =', LIGAND_INPUT)
print('LIGAND_BASENAME =', LIGAND_BASENAME)
print('FORCEFIELD =', FORCEFIELD)
print('WATER_MODEL =', WATER_MODEL)


## Instalacion base

Si el runtime cambia, vuelve a correr estas celdas desde aqui.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y gromacs openbabel
!pip -q install pdbfixer openmm biopython acpype pandas numpy matplotlib


In [ ]:
# Ajustes de librerias que ACPYPE/Amber puede requerir en Colab.
!ln -sf /usr/local/lib/python3.12/dist-packages/acpype/amber_linux/lib/libmfhdf.so.0.0.0 /usr/lib/libmfhdf.so.0 || true
!ln -sf /usr/local/lib/python3.12/dist-packages/acpype/amber_linux/lib/libdf.so.0.0.0 /usr/lib/libdf.so.0 || true
!ln -sf /usr/local/lib/python3.12/dist-packages/acpype/amber_linux/lib/libhdf5_hl.so.310.0.2 /usr/lib/libhdf5_hl.so.310 || true
!ln -sf /usr/local/lib/python3.12/dist-packages/acpype/amber_linux/lib/libhdf5.so.310.2.0 /usr/lib/libhdf5.so.310 || true
!ln -sf /usr/local/lib/python3.12/dist-packages/acpype/amber_linux/lib/libzip.so.5.5 /usr/lib/libzip.so.5 || true
!ldconfig
!ldd /usr/local/lib/python3.12/dist-packages/acpype/amber_linux/bin/teLeap | grep -E 'zip|hdf5|mfhdf|df' || true


## Receptor

Esta seccion limpia el PDB del receptor antes de pasarlo por PDBFixer. La limpieza elimina:
- `HETATM`
- hidrogenos preexistentes
- atomos duplicados por `(chain, resid, resname, atom_name)`


In [ ]:
seen = set()
kept = []

for line in RECEPTOR_PDB.read_text(encoding='utf-8', errors='ignore').splitlines():
    if line.startswith(('TER', 'END')):
        kept.append(line)
        continue
    if not line.startswith('ATOM'):
        continue
    atom_name = line[12:16].strip()
    resname = line[17:20].strip()
    chain = line[21].strip()
    resid = line[22:26].strip()
    element = line[76:78].strip()
    if atom_name.startswith('H') or element == 'H':
        continue
    key = (chain, resid, resname, atom_name)
    if key in seen:
        continue
    seen.add(key)
    kept.append(line)

PROTEIN_ONLY_PDB.write_text('\n'.join(kept) + '\n', encoding='utf-8')
print(PROTEIN_ONLY_PDB)
print('lineas:', len(kept))


In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename=str(PROTEIN_ONLY_PDB))
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()

with FIXED_PDB.open('w') as fh:
    PDBFile.writeFile(fixer.topology, fixer.positions, fh)

print(FIXED_PDB)


In [ ]:
%cd {RUN_DIR}
!printf "1\n1\n" | gmx pdb2gmx -f {FIXED_PDB.name} -o receptor_processed.gro -water {WATER_MODEL} -ff {FORCEFIELD} > pdb2gmx.log 2>&1
!tail -60 pdb2gmx.log
!ls -lh receptor_processed.gro topol.top


## Ligando con ACPYPE

Por defecto la libreta usa `ligand_input.mol2` y `alpelisib`.


In [ ]:
%cd {RUN_DIR}
!acpype -i {LIGAND_INPUT.name} -b {LIGAND_BASENAME} -c bcc -n 0
!ls -lh {LIGAND_ACPYPE_DIR}


## Ensamblado proteina + ligando

El script existente del repositorio trabaja con `--run_dir`. Aqui lo usamos y luego corregimos `topol.top` para que quede en orden GROMACS valido.


In [ ]:
%cd {RUN_DIR}
!cp {LIGAND_ACPYPE_DIR / (LIGAND_BASENAME + '_GMX.itp')} .
!cp {LIGAND_ACPYPE_DIR / ('posre_' + LIGAND_BASENAME + '.itp')} .
!python /content/drive/MyDrive/GEMAMX/gromacs/prepare_protein_ligand_system.py \
  --run_dir {RUN_DIR} \
  --protein_gro receptor_processed.gro \
  --protein_top topol.top \
  --ligand_gro {LIGAND_ACPYPE_DIR.name}/{LIGAND_BASENAME}_GMX.gro \
  --ligand_itp {LIGAND_ITP_NAME} \
  --ligand_posre {LIGAND_POSRE_NAME} \
  --ligand_name {LIGAND_BASENAME} \
  --output_gro complex.gro
!ls -lh complex.gro topol.top {LIGAND_ITP_NAME} {LIGAND_POSRE_NAME}


In [ ]:
import re

top = Path('topol.top')
itp = Path(LIGAND_ITP_NAME)
text = top.read_text(encoding='utf-8')

m = re.search(r'\[\s*moleculetype\s*\]\s*\n\s*;.*\n\s*(\S+)', itp.read_text(encoding='utf-8'))
ligand_name = m.group(1) if m else LIGAND_BASENAME

include_block = '\n'.join([
    '; Ligand parameters',
    f'#include "{LIGAND_ITP_NAME}"',
    '#ifdef POSRES_LIGAND',
    f'#include "{LIGAND_POSRE_NAME}"',
    '#endif',
    ''
])

lines = text.splitlines()
clean = []
skip_posre_block = False
seen_itp = False

for line in lines:
    stripped = line.strip()
    if skip_posre_block:
        if stripped == '#endif':
            skip_posre_block = False
        continue
    if stripped == f'#include "{LIGAND_ITP_NAME}"':
        if seen_itp:
            continue
        seen_itp = True
        continue
    if stripped in ('#ifdef POSRES_LIG', '#ifdef POSRES_LIGAND'):
        skip_posre_block = True
        continue
    clean.append(line)

text = '\n'.join(clean)
marker = f'#include "{FORCEFIELD}.ff/forcefield.itp"'
if include_block not in text:
    text = text.replace(marker, marker + '\n\n' + include_block, 1)

lines = text.splitlines()
out = []
in_molecules = False
ligand_written = False

for line in lines:
    stripped = line.strip()
    if stripped == '[ molecules ]':
        in_molecules = True
        out.append(line)
        continue
    if in_molecules and stripped and not stripped.startswith(';'):
        if stripped.split()[0].lower() == ligand_name.lower():
            if ligand_written:
                continue
            out.append(f'{ligand_name:<16} 1')
            ligand_written = True
            continue
    out.append(line)

if not ligand_written:
    out.append(f'{ligand_name:<16} 1')

top.write_text('\n'.join(out) + '\n', encoding='utf-8')
print('ligand_name =', ligand_name)


In [ ]:
!head -40 topol.top
!tail -20 topol.top


## Caja, agua e iones


In [ ]:
%cd {RUN_DIR}
!gmx editconf -f complex.gro -o boxed.gro -c -d 1.0 -bt cubic
!gmx solvate -cp boxed.gro -cs spc216.gro -o solvated.gro -p topol.top


In [ ]:
%%writefile ions.mdp
integrator = steep
emtol = 1000.0
emstep = 0.01
nsteps = 500
nstlist = 10
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
pbc = xyz


In [ ]:
!gmx grompp -f ions.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 2
!printf "SOL\n" | gmx genion -s ions.tpr -o solv_ions.gro -p topol.top -pname NA -nname CL -neutral


## Minimizacion y equilibrio


In [ ]:
%%writefile em.mdp
integrator  = steep
emtol       = 1000.0
emstep      = 0.01
nsteps      = 50000
nstlist     = 10
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb    = 1.0
rvdw        = 1.0
pbc         = xyz


In [ ]:
!gmx grompp -f em.mdp -c solv_ions.gro -p topol.top -o em.tpr -maxwarn 2
!gmx mdrun -deffnm em


In [ ]:
%%writefile nvt.mdp
title                   = NVT equilibration
integrator              = md
nsteps                  = 50000
dt                      = 0.002
nstxout-compressed      = 1000
nstenergy               = 500
nstlog                  = 500
continuation            = no
constraint_algorithm    = lincs
constraints             = h-bonds
cutoff-scheme           = Verlet
nstlist                 = 10
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
pbc                     = xyz
tcoupl                  = V-rescale
tc-grps                 = Protein non-Protein
tau_t                   = 0.1 0.1
ref_t                   = 300 300
pcoupl                  = no
gen_vel                 = yes
gen_temp                = 300
gen_seed                = -1


In [ ]:
!gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 2
!gmx mdrun -deffnm nvt


In [ ]:
%%writefile npt.mdp
title                   = NPT equilibration
integrator              = md
nsteps                  = 50000
dt                      = 0.002
nstxout-compressed      = 1000
nstenergy               = 500
nstlog                  = 500
continuation            = yes
constraint_algorithm    = lincs
constraints             = h-bonds
cutoff-scheme           = Verlet
nstlist                 = 10
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
pbc                     = xyz
tcoupl                  = V-rescale
tc-grps                 = Protein non-Protein
tau_t                   = 0.1 0.1
ref_t                   = 300 300
pcoupl                  = Parrinello-Rahman
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
gen_vel                 = no


In [ ]:
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 2
!gmx mdrun -deffnm npt


In [ ]:
%%writefile md.mdp
title                   = Production MD
integrator              = md
nsteps                  = 250000
dt                      = 0.002
nstxout-compressed      = 1000
nstenergy               = 1000
nstlog                  = 1000
continuation            = yes
constraint_algorithm    = lincs
constraints             = h-bonds
cutoff-scheme           = Verlet
nstlist                 = 20
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
pbc                     = xyz
tcoupl                  = V-rescale
tc-grps                 = Protein non-Protein
tau_t                   = 0.1 0.1
ref_t                   = 300 300
pcoupl                  = Parrinello-Rahman
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
gen_vel                 = no


In [ ]:
!gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 2
!gmx mdrun -deffnm md


## GPU y verificacion opcional


In [ ]:
!nvidia-smi
!gmx mdrun -version


In [ ]:
# Ejemplo de corrida usando GPU en Colab con Tesla T4.
# Ejecuta esto solo si quieres forzar GPU en NVT, NPT o MD.
# !gmx mdrun -deffnm nvt -nb gpu -pme gpu -bonded cpu -update cpu
